# 🎮 Day 5 Finale — Live Computer Vision Playground
### ML Bootcamp · Day 5 of 5

Time to play! Your **camera** + the models from Day 4 = live AI games.

| Game | What the AI does | Model |
|---|---|---|
| 🔍 **1. Scavenger hunt** | Finds objects you show it, live | YOLO11 detection |
| 🙌 **2. Hands-up challenge** | Tracks your body, counts your reps | YOLO11 pose |
| ✊✋✌️ **3. Rock–paper–scissors vs AI** | Reads your fingers | MediaPipe Hands |
| 🧙 **4. Invisibility cloak** | Cuts you out of the picture | YOLO11 segmentation |

**How it works:** your browser sends camera frames to Colab's computer, the model draws on them, and the result comes back — a few times per second.

> ⚡ **Runtime → Change runtime type → T4 GPU** (much smoother). Then run the cells **in order**.
>
> 📷 When your browser asks, click **Allow** for the camera. Click **Stop** (or the video) to end a game.
>
> 📱 **On a phone?** Open this notebook in **Chrome** (Android) or **Safari** (iPhone), choose *Desktop site* if Colab looks squashed, and use the 📸 *photo mode* cells if live video is slow.

---
## Setup (run once, ~1 minute)

In [ ]:
!pip install -q ultralytics mediapipe
!wget -q -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

In [ ]:
import cv2, numpy as np, base64, json, time, random, collections
import matplotlib.pyplot as plt
from IPython.display import display, Javascript, HTML
from ultralytics import YOLO

try:
    from google.colab.output import eval_js
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not in Colab: the live camera only works in Google Colab.")

detector = YOLO("yolo11n.pt")          # 80 everyday objects
poser    = YOLO("yolo11n-pose.pt")     # 17 body key points
segmenter = YOLO("yolo11n-seg.pt")     # exact outline of each object
print("Models ready!")

### The camera engine
This cell adds a small piece of **JavaScript** to the page: it opens your camera, sends a frame to Python, waits for the answer, shows it, and repeats. You don't need to understand it — but it's only ~40 lines, have a look!

In [ ]:
CAMERA_JS = r"""
var cv_video, cv_box, cv_img, cv_status, cv_stopped, cv_stream;

async function cv_start(facing) {
  cv_stopped = false;
  cv_box = document.createElement('div');
  cv_box.style.cssText = 'font-family:sans-serif; max-width:640px';
  cv_status = document.createElement('div');
  cv_status.style.cssText = 'font-size:22px; font-weight:bold; padding:8px; background:#13213A; color:#F7F5EF; border-radius:8px 8px 0 0';
  cv_status.innerHTML = 'Starting camera...';
  cv_img = document.createElement('img');
  cv_img.style.cssText = 'width:100%; display:block; background:#000; cursor:pointer';
  var btn = document.createElement('button');
  btn.textContent = 'Stop';
  btn.style.cssText = 'font-size:20px; padding:8px 24px; margin-top:6px; background:#F28C28; border:none; border-radius:8px; cursor:pointer';
  btn.onclick = cv_img.onclick = function () { cv_stopped = true; };
  cv_video = document.createElement('video');
  cv_video.setAttribute('playsinline', '');
  cv_video.muted = true;
  cv_video.style.display = 'none';
  cv_box.append(cv_status, cv_img, btn, cv_video);
  document.body.appendChild(cv_box);
  cv_stream = await navigator.mediaDevices.getUserMedia(
      {video: {facingMode: facing, width: {ideal: 640}, height: {ideal: 480}}, audio: false});
  cv_video.srcObject = cv_stream;
  await cv_video.play();
  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
}

function cv_grab() {
  var c = document.createElement('canvas');
  var scale = 640 / Math.max(cv_video.videoWidth, cv_video.videoHeight, 1);
  c.width = Math.round(cv_video.videoWidth * Math.min(scale, 1));
  c.height = Math.round(cv_video.videoHeight * Math.min(scale, 1));
  c.getContext('2d').drawImage(cv_video, 0, 0, c.width, c.height);
  return c.toDataURL('image/jpeg', 0.8);
}

async function cv_frame(imgUrl, status, facing) {
  if (!cv_video) { await cv_start(facing); }
  if (imgUrl) { cv_img.src = imgUrl; }
  if (status) { cv_status.innerHTML = status; }
  if (cv_stopped) {
    cv_stream.getTracks().forEach(function (t) { t.stop(); });
    cv_video.remove(); cv_video = null;
    return '';
  }
  return cv_grab();
}
"""

def to_jpeg_url(img_bgr, quality=80):
    ok, buf = cv2.imencode(".jpg", img_bgr, [cv2.IMWRITE_JPEG_QUALITY, quality])
    return "data:image/jpeg;base64," + base64.b64encode(buf).decode()

def from_url(url):
    data = base64.b64decode(url.split(",")[1])
    return cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)

def run_live(process, facing="user", mirror=True, max_seconds=180):
    """Run a game live. process(frame) must return (drawn_frame, status_html)."""
    if not IN_COLAB:
        print("Live camera needs Google Colab."); return
    display(Javascript(CAMERA_JS))
    reply = eval_js("cv_frame('', 'Starting camera...', %s)" % json.dumps(facing))
    start, frames = time.time(), 0
    while reply and time.time() - start < max_seconds:
        frame = from_url(reply)
        if mirror:
            frame = cv2.flip(frame, 1)                 # selfie view: move left, image moves left
        out, status = process(frame)
        frames += 1
        fps = frames / (time.time() - start)
        reply = eval_js("cv_frame(%s, %s, %s)" % (json.dumps(to_jpeg_url(out)),
                        json.dumps(f"{status} <span style='float:right; font-weight:normal'>{fps:.1f} fps</span>"),
                        json.dumps(facing)))
    if reply:   # time limit reached: close the camera
        eval_js("cv_stopped = true; cv_frame('', 'Time up!', %s)" % json.dumps(facing))
    print(f"Stopped after {frames} frames ({frames / max(time.time() - start, 1e-6):.1f} frames per second).")

def take_photo(facing="user"):
    """Photo mode: one picture, for slow connections and phones."""
    if not IN_COLAB:
        print("Photo mode needs Google Colab."); return None
    display(Javascript(CAMERA_JS))
    reply = eval_js("cv_frame('', 'Smile! Taking photo in 2 s...', %s)" % json.dumps(facing))
    eval_js("new Promise(r => setTimeout(r, 2000))")
    reply = eval_js("cv_frame('', 'Got it!', %s)" % json.dumps(facing))
    eval_js("cv_stopped = true; cv_frame('', 'Photo taken - camera off', %s)" % json.dumps(facing))
    return from_url(reply) if reply else None

def show(img_bgr, title=""):
    plt.figure(figsize=(9, 7)); plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title, fontsize=16); plt.axis("off"); plt.show()

def big_text(img, text, pos=(20, 60), scale=1.6, color=(255, 255, 255), bg=(19, 33, 58)):
    (w, h), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 3)
    x, y = pos
    cv2.rectangle(img, (x - 10, y - h - 14), (x + w + 10, y + 14), bg, -1)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, 3, cv2.LINE_AA)

print("Camera engine ready!")

In [ ]:
# 📷 Camera test: should show your face with a big "Hello!" (click Stop when done)
def hello(frame):
    big_text(frame, "Hello, ML Bootcamp!", (20, 60), 1.2)
    return frame, "Camera works! Click Stop."

run_live(hello)

---
## 🔍 Game 1 · Scavenger hunt (YOLO object detection)
The AI picks **5 random objects**. Find them around you and show them to the camera before time runs out!
YOLO11 knows 80 everyday things: person, cell phone, bottle, cup, book, laptop, backpack, chair, scissors, keyboard…

> 📱 Phone users: use the **back camera** — change `facing="user"` to `facing="environment"` and `mirror=False`.

In [ ]:
CANDIDATES = ["cell phone", "bottle", "cup", "book", "laptop", "backpack", "chair", "scissors",
              "keyboard", "mouse", "handbag", "clock", "remote", "umbrella", "tie", "banana", "apple", "spoon"]

def make_hunt(n_items=5, seconds=90, conf=0.45):
    targets = random.sample(CANDIDATES, n_items)
    found = {}
    start = time.time()
    def process(frame):
        left = max(0, seconds - (time.time() - start))
        r = detector(frame, conf=conf, verbose=False)[0]
        out = r.plot(line_width=3, font_size=18)
        seen = {r.names[int(c)] for c in r.boxes.cls}
        for t in targets:
            if t in seen and t not in found and left > 0:
                found[t] = round(seconds - left, 1)          # time when it was found
        score = len(found)
        if score == len(targets):
            big_text(out, f"YOU WIN! {max(found.values()):.0f} s", (20, 70), 1.6, bg=(40, 140, 40))
        elif left == 0:
            big_text(out, f"TIME UP! {score}/{len(targets)}", (20, 70), 1.6, bg=(30, 30, 200))
        else:
            big_text(out, f"{left:4.0f} s   {score}/{len(targets)}", (20, 60), 1.4)
        checklist = " &nbsp; ".join(("✅ " if t in found else "⬜ ") + t for t in targets)
        return out, checklist
    return process, targets, found

hunt, targets, found = make_hunt()
print("Find these:", targets)
run_live(hunt, facing="user", mirror=True)
print("You found:", found)

🧠 **Think about it:** when does YOLO fail? Try: the object half-hidden, far away, in bad light, or upside down. What does it call things it **doesn't** know (a pen, a calculator, a keke)?

✍️ **Challenge:** set `conf=0.25` in `make_hunt(...)`. Is it easier? Do you get more wrong boxes? (Day 2's precision vs recall!)

---
## 🙌 Game 2 · Hands-up challenge (YOLO pose)
YOLO-pose finds **17 key points** on every person. We count a **rep** every time **both wrists go above your nose** and come back down.
How many reps in **30 seconds**? Stand back so your whole upper body is in view.

In [ ]:
NOSE, L_WRIST, R_WRIST = 0, 9, 10

def make_hands_up(seconds=30):
    state = {"up": False, "reps": 0, "start": None}
    def process(frame):
        if state["start"] is None:
            state["start"] = time.time()
        left = max(0, seconds - (time.time() - state["start"]))
        r = poser(frame, verbose=False)[0]
        out = r.plot(boxes=False, line_width=3)
        if r.keypoints is not None and len(r.keypoints) > 0 and left > 0:
            kp = r.keypoints.xy[0].cpu().numpy()          # first person
            conf = r.keypoints.conf[0].cpu().numpy()
            if min(conf[[NOSE, L_WRIST, R_WRIST]]) > 0.5:
                both_up = kp[L_WRIST][1] < kp[NOSE][1] and kp[R_WRIST][1] < kp[NOSE][1]   # y grows downwards!
                if both_up and not state["up"]:
                    state["reps"] += 1
                state["up"] = both_up
        if left > 0:
            big_text(out, f"REPS: {state['reps']}   {left:.0f} s", (20, 60), 1.5)
            msg = "🙌 Hands UP!" if state["up"] else "⬇️ Hands down"
        else:
            big_text(out, f"FINAL: {state['reps']} reps!", (20, 70), 1.7, bg=(40, 140, 40))
            msg = "Time up! Click Stop."
        return out, msg
    return process, state

game, state = make_hands_up(seconds=30)
run_live(game)
print("Your score:", state["reps"], "reps")

🧠 **Why `<` and not `>`?** In images, **y = 0 is the top** — so "higher up" means a *smaller* y.

✍️ **Challenge:** turn it into a **squat counter**: count a rep when your hips (key points 11 and 12) drop below your knees (13 and 14). The 17 key points are: 0 nose, 1–2 eyes, 3–4 ears, 5–6 shoulders, 7–8 elbows, 9–10 wrists, 11–12 hips, 13–14 knees, 15–16 ankles.

---
## ✊✋✌️ Game 3 · Rock–paper–scissors vs the AI (MediaPipe Hands)
**MediaPipe** is Google's toolkit for fast, on-device vision: hands, faces, body pose, gestures. It runs on phones, laptops and even in web pages.

Its hand model finds **21 points** on each hand. From them we work out **which fingers are up**:
- ✊ 0 fingers = **rock** · ✋ 5 fingers = **paper** · ✌️ index + middle = **scissors**

In [ ]:
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

hand_model = mp_vision.HandLandmarker.create_from_options(mp_vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path="hand_landmarker.task"), num_hands=2))

HAND_LINKS = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),(5,9),(9,10),(10,11),(11,12),
              (9,13),(13,14),(14,15),(15,16),(13,17),(17,18),(18,19),(19,20),(0,17)]

def fingers_up(pts):
    """pts: 21 (x, y) points. A finger is 'up' if its tip is further from the wrist than its middle joint."""
    d = lambda a, b: np.linalg.norm(pts[a] - pts[b])
    thumb = d(4, 9) > 0.8 * d(2, 9)                      # thumb tip far from the middle of the palm
    others = [d(tip, 0) > d(pip, 0) for tip, pip in [(8, 6), (12, 10), (16, 14), (20, 18)]]
    return [thumb] + others                              # [thumb, index, middle, ring, pinky]

def find_hands(frame, draw=True):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hand_model.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    h, w = frame.shape[:2]
    hands = []
    for lm in result.hand_landmarks:
        pts = np.array([[p.x * w, p.y * h] for p in lm])
        if draw:
            for a, b in HAND_LINKS:
                cv2.line(frame, tuple(pts[a].astype(int)), tuple(pts[b].astype(int)), (255, 255, 255), 3)
            for x, y in pts.astype(int):
                cv2.circle(frame, (x, y), 6, (40, 140, 242), -1)
        hands.append(pts)
    return hands

def gesture(up):
    if sum(up) == 0 or up == [True, False, False, False, False]:
        return "rock"
    if sum(up) >= 4:
        return "paper"
    if up[1] and up[2] and not up[3] and not up[4]:
        return "scissors"
    return None

In [ ]:
# Warm-up: a live finger counter
def finger_counter(frame):
    hands = find_hands(frame)
    total = sum(sum(fingers_up(pts)) for pts in hands)
    big_text(frame, f"Fingers: {total}", (20, 70), 2.0)
    return frame, f"{len(hands)} hand(s) · {total} finger(s) up"

run_live(finger_counter)

In [ ]:
EMOJI = {"rock": "✊", "paper": "✋", "scissors": "✌️"}
BEATS = {"rock": "scissors", "paper": "rock", "scissors": "paper"}

def make_rps(rounds=5):
    s = {"you": 0, "ai": 0, "round": 1, "t0": time.time(), "result": ""}
    def process(frame):
        hands = find_hands(frame)
        mine = gesture(fingers_up(hands[0])) if hands else None
        t = time.time() - s["t0"]
        if s["round"] > rounds:
            winner = "YOU WIN!" if s["you"] > s["ai"] else ("AI WINS!" if s["ai"] > s["you"] else "DRAW!")
            big_text(frame, f"{winner} {s['you']}-{s['ai']}", (20, 70), 1.8, bg=(40, 140, 40))
            return frame, "Game over! Click Stop."
        if t < 3:                                         # countdown
            big_text(frame, f"Round {s['round']}:  {3 - int(t)}", (20, 70), 1.8)
            return frame, f"Get ready... You {s['you']} - {s['ai']} AI &nbsp; {s['result']}"
        if t < 3.5 or mine is None:                       # SHOOT: read your hand
            big_text(frame, "SHOOT!", (20, 70), 2.2, bg=(30, 30, 200))
            return frame, "Show rock, paper or scissors!"
        ai = random.choice(list(BEATS))
        if BEATS[mine] == ai:
            s["you"] += 1; verdict = "you win the round"
        elif BEATS[ai] == mine:
            s["ai"] += 1; verdict = "AI wins the round"
        else:
            verdict = "draw"
        s["result"] = f"Last: you {EMOJI[mine]} vs AI {EMOJI[ai]} → {verdict}"
        s["round"] += 1; s["t0"] = time.time()
        big_text(frame, f"You: {mine}  AI: {ai}", (20, 70), 1.3)
        return frame, s["result"]
    return process, s

rps, score = make_rps(rounds=5)
run_live(rps)
print(f"Final score: you {score['you']} - {score['ai']} AI")

🧠 **Is this game fair?** The AI chooses *randomly* — it doesn't look at your hand first. How could you make an AI that **cheats**? (Easy: read your hand, then pick the winner!) What about one that learns *your habits*? That's machine learning on your behaviour.

✍️ **Challenge:** add a 4th gesture: 👍 **thumbs up** = `[True, False, False, False, False]`... but right now that counts as rock! Change `gesture()` so a thumbs-up prints "LIKE 👍" instead.

---
## 🧙 Game 4 · The invisibility cloak (YOLO segmentation)
Segmentation finds the **exact pixels** of every person. If we replace those pixels with a photo of the **empty room**, you disappear!

1. Run the cell and **step out of the camera's view** for the first 4 seconds (it photographs the empty background).
2. Walk back in… and you're invisible. 🫥 Hold the phone/laptop **still**.

In [ ]:
def make_cloak(wait=4, mode="invisible"):
    s = {"bg": None, "t0": time.time()}
    def process(frame):
        t = time.time() - s["t0"]
        if s["bg"] is None:
            if t < wait:
                show_frame = frame.copy()
                big_text(show_frame, f"Step out! {wait - int(t)}", (20, 70), 1.8, bg=(30, 30, 200))
                return show_frame, "Leave the picture so I can see the empty room..."
            s["bg"] = frame.copy()
        r = segmenter(frame, classes=[0], verbose=False, retina_masks=True)[0]    # class 0 = person
        if r.masks is None:
            return frame, "Come back in... 👀"
        mask = r.masks.data.cpu().numpy().max(axis=0)                            # all people together
        mask = cv2.resize(mask, (frame.shape[1], frame.shape[0]))
        mask = cv2.GaussianBlur(mask, (21, 21), 0)[..., None]                   # soft edges
        if mode == "invisible":
            out = (mask * s["bg"] + (1 - mask) * frame).astype(np.uint8)
        else:   # "blur": blur the background like a video call
            blurred = cv2.GaussianBlur(frame, (55, 55), 0)
            out = (mask * frame + (1 - mask) * blurred).astype(np.uint8)
        return out, f"{len(r.masks)} person(s) {'hidden 🫥' if mode == 'invisible' else 'in focus'}"
    return process

run_live(make_cloak(mode="invisible"))

In [ ]:
# Video-call style background blur (no need to step out)
run_live(make_cloak(wait=0, mode="blur"))

🧠 Where do you see this every day? **Zoom / Google Meet background blur**, TikTok and Snapchat filters, photo "portrait mode". Same idea: segmentation, then mix pixels.

---
## 📸 Photo mode (for slow connections and phones)
If live video is too slow, take **one photo** and run any model on it.

In [ ]:
photo = take_photo(facing="user")      # phones: facing="environment" for the back camera
if photo is not None:
    show(detector(photo, verbose=False)[0].plot(), "Objects")
    show(poser(photo, verbose=False)[0].plot(), "Pose")
    with_hands = photo.copy(); hands = find_hands(with_hands)
    show(with_hands, f"Hands: {[sum(fingers_up(h)) for h in hands]} fingers up")

---
## 🚀 Where to go from here
- **MediaPipe** has more ready models: **face mesh** (478 face points — make your own filters), **gesture recognizer** (👍 ✌️ 👆 built in), **pose landmarker** (33 body points), **selfie segmentation**. They also run **inside a web page** with JavaScript, right on your phone. Try Google's official gesture-recognizer demo: https://codepen.io/mediapipe-preview/pen/zYamdVd
- **YOLO** can be **trained on your own objects** (Day 4, transfer learning): naira notes, local fruits, keke napep, faulty components on a circuit board…
- Put it on a **Raspberry Pi + camera** (Day 5, Part 2) and you have a real smart camera.

✍️ **Mini-project ideas:** a sign-language letter reader · an exam-hall "phone detector" · a squat coach that checks your knee angle · a hand-controlled slideshow (swipe left/right).

**Thank you for 5 days of machine learning! 🎉**